In [1]:
import sys
sys.path.append("..")
import numpy as np
import pygeode as pyg
from matplotlib import pyplot as plt
import xarray as xr
from pydmd import DMD, BOPDMD, HankelDMD
from pydmd.preprocessing import hankel_preprocessing
from matplotlib import colors
from matplotlib.colors import LogNorm, Normalize
import scipy.signal
from scipy.ndimage import gaussian_filter
import pandas as pd
from pathlib import Path

%matplotlib widget

In [2]:
# ----------------------------
# Load data for E3SM present day interactive
# ----------------------------
model = "E3SM_PDINT"
PDINT_data_path = Path("/gws/nopw/j04/qboi/quoca/phase1/E3SMv3/QUOCA/LLNL/E3SM/PDINT/r1i1p1f1/monZ/")
variables = ["ta", "ua", "va", "wap", "o3"]

PDINT_dss = {}

all_files = []
for var in variables:
    files = sorted((PDINT_data_path / var).glob("**/*_??????-??????.nc"))
    if not files:
        raise FileNotFoundError(f"No files found for var={var} under {PDINT_data_path/var}")
    all_files.extend(files)
    
PDINT_ds = xr.open_mfdataset(
    all_files,
    combine="by_coords",
    decode_times=True,
    use_cftime=True,
)

PDINT_ds = PDINT_ds.rename({
    "ua": "u",
    "va": "v",
    "ta": "t",
    "wap": "w",
    "plev": "pres",
})

PDINT_ds = PDINT_ds.transpose("pres", "lat", "time", ...)

PDINT_ds = PDINT_ds.drop_vars(['lat_bnds','time_bnds'])
PDINT_ds = PDINT_ds.sortby("time")
PDINT_ds = PDINT_ds.assign_coords(
    pres=("pres", PDINT_ds.pres.values / 100.0)
)
PDINT_ds["pres"].attrs["units"] = "hPa"
PDINT_ds = PDINT_ds.sortby('pres')
PDINT_ds = PDINT_ds.sortby('lat')

print(PDINT_ds)

<xarray.Dataset>
Dimensions:  (time: 1080, pres: 42, lat: 180)
Coordinates:
  * time     (time) object 0011-01-16 12:00:00 ... 0100-12-16 12:00:00
  * pres     (pres) float64 0.4 0.5 0.7 1.0 1.5 ... 700.0 850.0 925.0 1e+03
  * lat      (lat) float64 -89.5 -88.5 -87.5 -86.5 -85.5 ... 86.5 87.5 88.5 89.5
Data variables:
    o3       (pres, lat, time) float32 dask.array<chunksize=(42, 180, 1080), meta=np.ndarray>
    t        (pres, lat, time) float32 dask.array<chunksize=(42, 180, 1080), meta=np.ndarray>
    u        (pres, lat, time) float32 dask.array<chunksize=(42, 180, 1080), meta=np.ndarray>
    v        (pres, lat, time) float32 dask.array<chunksize=(42, 180, 1080), meta=np.ndarray>
    w        (pres, lat, time) float32 dask.array<chunksize=(42, 180, 1080), meta=np.ndarray>
Attributes: (12/50)
    Conventions:             CF-1.8 QUOCA
    activity_id:             QUOCA
    activity_participation:  QUOCA
    branch_method:           standard
    branch_time_in_child:    0.0
    bra

In [3]:
# ----------------------------
# Config options
# ----------------------------


# NEED TO CHECK IF THERE IS TREND IN THE MODEL AND ALSO WHAT THE DATA OF THE MODEL LOOKS LIKE (HOW MANY PRESSURE AND LATITUDE LEVELS TO SEE IF I NEED SMOOTHING)

clim = False
anom = True
detrend = False
smooth = True
split_time = False

config_str=[]
config_str.append(model)

if anom:
    config_str.append('anom')
if clim:
    config_str.append('clim')
if detrend:
    config_str.append('detrend')
if smooth:
    config_str.append('smooth')

config_str = '_'.join(config_str)
print(config_str)
data_file_out = '/home/users/mm2947/jupyter/processed_data/E3SM/'+config_str+'.nc'

E3SM_PDINT_anom_smooth


In [4]:
# ----------------------------
# Select pressure and latitude range
# ----------------------------
pres_min = 10       # hPa
pres_max = 150      # hPa
lat_min = -35
lat_max = 35

PDINT_ds = PDINT_ds.sel(pres=slice(pres_min, pres_max), lat=slice(lat_min, lat_max))

In [5]:
# ----------------------------
# Monthly average
# ----------------------------
# ds = ds.resample(time="MS").mean()

In [6]:
# ----------------------------
# Remove climatology
# ----------------------------

if clim:
    PDINT_ds_clim = PDINT_ds.groupby("time.month").mean("time")
    PDINT_ds = PDINT_ds.groupby("time.month") - PDINT_ds_clim
    PDINT_ds = PDINT_ds.reset_coords("month", drop=True)

In [7]:
# ----------------------------
# Detrend
# ----------------------------

def detrend_xr(da):
    t = xr.DataArray(np.arange(da.sizes["time"]), dims="time", coords={"time": da["time"]})
    p = da.polyfit(dim="time", deg=1)
    trend = xr.polyval(t, p.polyfit_coefficients)
    return da - trend

if detrend:
    for v in PDINT_ds.data_vars.keys():
        PDINT_ds[v] = detrend_xr(PDINT_ds[v])

In [8]:
# ----------------------------
# Compute time anomaly
# ----------------------------

if anom:
    for v in PDINT_ds.data_vars.keys():
        PDINT_ds[v] -= PDINT_ds[v].mean(dim="time")

In [9]:
# ----------------------------
# Laplacian smoothing
# ----------------------------

sigma_lat  = 1.0
sigma_pres = 0.7

if smooth:
    for v in PDINT_ds.data_vars.keys():
        PDINT_ds[v] = xr.apply_ufunc(
        gaussian_filter,
        PDINT_ds[v],
        kwargs=dict(
            sigma=(0.0, sigma_lat, sigma_pres),
            mode="nearest"
        ),
        input_core_dims=[["time", "lat", "pres"]],
        output_core_dims=[["time", "lat", "pres"]],
        vectorize=False,
        dask="parallelized",
        dask_gufunc_kwargs={"allow_rechunk": True},
        output_dtypes=[PDINT_ds[v].dtype],
    )

In [ ]:
# ----------------------------
# Split 1x90 run into 3x30 runs
# ----------------------------

def split_into_n_year_blocks(ds, years_per_block=30, n_blocks=3, time_dim="time"):
    """
    Splits ds into n_blocks consecutive chunks, each spanning years_per_block years,
    based on the dataset's time coordinate (works with cftime calendars).
    """
    t = ds[time_dim]
    years = t.dt.year

    y0 = int(years.min().item())
    blocks = []
    for i in range(n_blocks):
        y_start = y0 + i * years_per_block
        y_end   = y_start + years_per_block - 1
        dsi = ds.sel({time_dim: (years >= y_start) & (years <= y_end)})
        blocks.append(dsi)
    return blocks

# Build base output path from your existing convention
out_path = Path(data_file_out)                    # e.g. .../E3SM/E3SM_anom_smooth.nc
out_dir  = out_path.parent
stem     = out_path.stem                          # e.g. "E3SM_anom_smooth"
suffix   = out_path.suffix                        # ".nc"
out_dir.mkdir(parents=True, exist_ok=True)

if split_time:
    blocks = split_into_n_year_blocks(PDINT_ds, years_per_block=30, n_blocks=3, time_dim="time")

    for i, dsi in enumerate(blocks, start=1):
        fn = out_dir / f"{stem}_{i}{suffix}"      # .../E3SM_anom_smooth_1.nc etc.
        dsi.to_netcdf(fn)
        print(f"Saved: {fn} | years {int(dsi.time.dt.year.min())}-{int(dsi.time.dt.year.max())} | nt={dsi.sizes['time']}")
else:
    PDINT_ds.to_netcdf(out_path)
    print(f"Saved: {out_path}")


In [ ]:
# ----------------------------
# File saving
# ----------------------------

#PDINT_ds.to_netcdf(data_file_out)